# Lesson 18 - Practice: HTTP & APIs with urllib

Work in `lessons/lesson-18-http-api/` (point a terminal at that folder).

Before you start, read `lesson.md` - every construct used in these exercises is
taught there. The clients you write talk to one tiny local server,
`local_api.py`, which runs on `127.0.0.1` on a free port. No internet access is
needed for anything in this lesson.

How to work through this notebook:

1. Read the **Task**, **File tree**, **Interface contract** and **Constraints**.
2. Create each file with a source editor (VS Code/Jupyter editor).
3. Run the **Driver** cell - it runs the same calls and pins the expected output.
4. Compare the Driver output with **Expected** before moving on.

The pytest Driver cells print only the stable summary line (`N passed in 0.01s`).
In your terminal you will also see progress dots and real timing - cosmetics,
ignore them. The other Driver cells print the exact program output.

E1 -> M3 build on each other, and they all import from files you wrote earlier;
do them in order. A wrong Driver output usually means a file from an earlier
exercise is off - fix it first, then continue.


## E1 - Split a URL into its parts

**Type:** REAL-WORLD - easy - writes 1 file

**Task**
A URL is four pieces: scheme, host, path, query. Write `split_url(url)` that
unpacks them with `urllib.parse.urlsplit`.

**File tree**

```
url_parts.py          (you create)
```

**Interface contract**
`split_url(url: str) -> tuple[str, str, str, str]` - returns
`(scheme, netloc, path, query)` in that order.

**Constraints**
- Use `urlsplit` from `urllib.parse`; return exactly those four fields.
- Driver imports `import url_parts as up`.
- Command: none - the Driver cell calls it directly.

**Expected** (Driver cell):
```
https | api.example.com | /v1/items | q=coffee%20latte&max=5
http | 127.0.0.1:1234 | /weather | city=tokyo
ftp | files.local | /report.csv | 
```
Note: `netloc` is host **plus port**; the query is everything after `?`.


In [1]:
import url_parts as up

urls = [
    "https://api.example.com/v1/items?q=coffee%20latte&max=5",
    "http://127.0.0.1:1234/weather?city=tokyo",
    "ftp://files.local/report.csv",
]
for url in urls:
    print(" | ".join(up.split_url(url)))

https | api.example.com | /v1/items | q=coffee%20latte&max=5
http | 127.0.0.1:1234 | /weather | city=tokyo
ftp | files.local | /report.csv | 


## E2 - Read the status code and its family

**Type:** REAL-WORLD - easy - writes 1 file

**Task**
Every response carries a status code - the verdict. Write `get_status` and
`status_family` so a caller can ask "what did the server say?" and
"is that a 2xx or a 4xx?".

**File tree**

```
client_basics.py      (you create)
```

**Interface contract**
- `get_status(base: str, path: str) -> int` - the response status code.
- `status_family(code: int) -> str` - one of `"2xx"`, `"3xx"`, `"4xx"`,
  `"5xx"`.

**Constraints**
- `get_status` must still return the code when the server answers a 4xx/5xx:
  catch `urllib.error.HTTPError` and return `exc.code`.
- Driver imports `import client_basics as cb` and `import local_api`, then
  loops over the paths shown in **Expected**.
- Command: none - the Driver cell calls it directly.

**Expected** (Driver cell):
```
200 2xx  /items
404 4xx  /items/999
405 4xx  /echo
400 4xx  /search
405 4xx  /count
404 4xx  /items/latte
```


In [6]:
import client_basics as cb
import local_api

api = local_api.LocalAPI()
try:
    for path in ["/items", "/items/999", "/echo", "/search", "/count", "/items/latte"]:
        code = cb.get_status(api.base, path)
        print(f"{code} {cb.status_family(code)}  {path}")
finally:
    api.stop()

200 2xx  /items
404 4xx  /items/999
405 4xx  /echo
400 4xx  /search
405 4xx  /count
404 4xx  /items/latte


## E3 - Read a JSON body

**Type:** REAL-WORLD - easy - edits 1 file

**Task**
`/items` answers with a JSON object. Write `get_json(base, path)` that returns
the parsed body as a dict.

**File tree**

```
client_basics.py      (edit: add get_json)
```

**Interface contract**
`get_json(base: str, path: str) -> dict` - decode `resp.read()` as UTF-8, then
`json.loads`.

**Constraints**
- Use `with urllib.request.urlopen(...) as resp:` so the response is closed.
- Keep both earlier functions in the file.
- Command: none - the Driver cell calls it directly.

**Expected** (Driver cell):
```
count = 4
first = coffee  last = latte
top price = 4.1
```


In [1]:
import client_basics as cb
import local_api

api = local_api.LocalAPI()
try:
    data = cb.get_json(api.base, "/items")
    print("count =", data["count"])
    print("first =", data["items"][0]["name"], " last =", data["items"][-1]["name"])
    print("top price =", max(item["price"] for item in data["items"]))
finally:
    api.stop()

count = 4
first = coffee  last = latte
top price = 4.1


## E4 - Inspect response headers

**Type:** REAL-WORLD - easy - edits 1 file

**Task**
`resp.headers` is a header map. Write `get_headers` that reports the
`Content-Type`, the declared `Content-Length`, and how many body bytes were
actually read.

**File tree**

```
client_basics.py      (edit: add get_headers)
```

**Interface contract**
`get_headers(base: str, path: str) -> dict` with keys:
- `"content-type"`   - from the header,
- `"content-length"` - from the header,
- `"body_bytes"`     - `len(raw)` of the bytes read.

**Constraints**
- Read the body into a variable once, then use it for the byte count.
- Keep all three earlier functions.
- Command: none - the Driver cell calls it directly.

**Expected** (Driver cell):
```
content-type: application/json; charset=utf-8
content-length header: 242
body_bytes: 242
```


In [1]:
import client_basics as cb
import local_api

api = local_api.LocalAPI()
try:
    info = cb.get_headers(api.base, "/items")
    print("content-type:", info["content-type"])
    print("content-length header:", info["content-length"])
    print("body_bytes:", info["body_bytes"])
finally:
    api.stop()

content-type: application/json; charset=utf-8
content-length header: 242
body_bytes: 242


## E5 - Handle a timeout

**Type:** REAL-WORLD - easy - edits 1 file

**Task**
`/slow` sleeps ~0.5s. Write `probe_timeout` that asks with a tiny timeout and
reports what the request layer raises. On Python 3.12 a read timeout raises
`TimeoutError` directly - it is NOT wrapped in `URLError`.

**File tree**

```
client_basics.py      (edit: add probe_timeout)
```

**Interface contract**
`probe_timeout(base: str, path: str = "/slow", timeout: float = 0.05) -> str`
- returns `f"TimeoutError {exc}"` when a `TimeoutError` is caught.

**Constraints**
- Catch `TimeoutError` only.
- Keep all four earlier functions.
- Command: none - the Driver cell calls it directly.

**Expected** (Driver cell):
```
TimeoutError timed out
```


In [1]:
import client_basics as cb
import local_api

api = local_api.LocalAPI()
try:
    print(cb.probe_timeout(api.base))
finally:
    api.stop()

TimeoutError timed out


## M1 - Categorize errors with APIError

**Type:** REAL-WORLD - medium - writes 1 file

**Task**
Two very different failures look the same to a naive client: "the server
answered with an error" vs "nothing answered at all". Write `fetch_json` which
maps each failure to a category: `status` (server answered 4xx/5xx) or
`timeout` (nothing came back in time).

**File tree**

```
api_client.py         (you create)
```

**Interface contract**
- `class APIError(Exception)` with attributes `category: str`,
  `status: int | None`, `detail: str | None`.
- `fetch_json(base: str, path: str, timeout: float = 3.0) -> dict` - parses the
  JSON body on success; raises `APIError` otherwise.
- `APIError("status", status=exc.code, detail=...)` for `HTTPError` - the
  `detail` is the `"error"` field of the JSON error body.
- `APIError("timeout", detail=str(exc))` for `TimeoutError`.

**Constraints**
- Order the `except` clauses carefully: `HTTPError` is a subclass of
  `URLError`. If the `URLError` handler comes first, every 4xx/5xx is
  misclassified as a network error. (You will meet this exact bug in H1.)
- No broad `except Exception`.
- Command: none - the Driver cell calls it directly.

**Expected** (Driver cell):
```
ok count = 4
APIError: status 404 item not found
APIError: timeout | timed out
```


In [1]:
import api_client as ac
import local_api

api = local_api.LocalAPI()
try:
    ok = ac.fetch_json(api.base, "/items")
    print("ok count =", ok["count"])
    try:
        ac.fetch_json(api.base, "/items/999")
    except ac.APIError as exc:
        print("APIError:", exc.category, exc.status, exc.detail)
    try:
        ac.fetch_json(api.base, "/slow", timeout=0.05)
    except ac.APIError as exc:
        print("APIError:", exc.category, "|", exc.detail)
finally:
    api.stop()

ok count = 4
APIError: status 404 item not found
APIError: timeout | timed out


## M2 - POST JSON to an API

**Type:** REAL-WORLD - medium - writes 1 file

**Task**
Up to now everything has been a GET. Write a small POST client that takes JSON,
sends it as the request body, and returns the status plus the parsed reply.

**File tree**

```
post_client.py        (you create)
```

**Interface contract**
`post_json(base: str, path: str, payload: dict | None = None,
           timeout: float = 3.0) -> tuple[int, dict]`
- returns `(resp.status, parsed body)`.

**Constraints**
- Serialize the body with `json.dumps(payload).encode("utf-8")`.
- Build `urllib.request.Request(base + path, data=..., method="POST")` and set
  the header `Content-Type: application/json` when a body is present.
- Gotcha: with NO body (`data=None`), urllib silently sends a **GET** unless
  `method="POST"` is passed explicitly. Pass `method="POST"` in every call,
  body or not.
- **Do not touch local_api.py** - `POST /count` refusing GET with 405 is
  intentional.
- Command: none - the Driver cell calls it directly.

**Expected** (Driver cell):
```
counts: [1, 2, 3]
echo status: 201
echo body: {'received': {'x': 1}}
```


In [1]:
import local_api
import post_client as pc

api = local_api.LocalAPI()
try:
    counts = [pc.post_json(api.base, "/count")[1]["n"] for _ in range(3)]
    print("counts:", counts)
    status, body = pc.post_json(api.base, "/echo", {"x": 1})
    print("echo status:", status)
    print("echo body:", body)
finally:
    api.stop()

counts: [1, 2, 3]
echo status: 201
echo body: {'received': {'x': 1}}


## M3 - Build a query string with urlencode

**Type:** REAL-WORLD - medium - writes 1 file

**Task**
Query strings have their own encoding rules: spaces become `+`, special
characters are percent-encoded. Write `search` that builds the query with
`urlencode` instead of hand-crafting strings.

**File tree**

```
search_client.py      (you create)
```

**Interface contract**
`search(base: str, term: str, max_items: int = 10, timeout: float = 3.0) -> dict`
- returns the parsed JSON body of `/search?q=..&max=..`.

**Constraints**
- Build the query with `urllib.parse.urlencode({"q": term, "max": max_items})`.
- The server treats `max` as an int; pass it as `max_items`.
- Command: none - the Driver cell calls it directly.

**Expected** (Driver cell):
```
'latte': count=1 items=['latte']
'coffee tea': count=0 items=[]
'muffin': count=1 items=['muffin']
```


In [2]:
import local_api
import search_client as sc

api = local_api.LocalAPI()
try:
    for term in ["latte", "coffee tea", "muffin"]:
        data = sc.search(api.base, term)
        names = [item["name"] for item in data["items"]]
        print(f"{term!r}: count={data['count']} items={names}")
finally:
    api.stop()

'latte': count=1 items=['latte']
'coffee tea': count=0 items=[]
'muffin': count=1 items=['muffin']


## H1 - Debug drill: fix the buggy weather client

**Type:** DEBUG - hard - edits 1 file

**Task**
`weather_client.py` is provided **with four planted bugs**. Find and fix all
four so the suite goes green. The test file and the server are correct - you
may read them but must not modify them.

**File tree**

```
weather_client.py     (provided, buggy - YOU fix it)
test_weather.py       (provided - do NOT modify)
local_api.py          (provided - do NOT modify)
```

**Hints**
- Bug 1: the traceback for `tokyo` points at JSON decoding - but the response
  was fine. Read the body once, then decode once.
- Bug 2: `new york` has a space. A URL query string must be percent-encoded;
  you already wrote `urlencode` in M3.
- Bug 3: `atlantis` must raise with `status == 404`, but the client reports it
  as a generic network error. Which except clause catches it first?
- Bug 4: `tokyo` decodes fine but the client reads a key the server never
  sends. Compare dict keys against `WEATHER` in `local_api.py`.

**Bug report** (fill in as you go):

```
# | Bug (what was wrong)           | Fix (what you changed)
--|--------------------------------|----------------------------
1 |                                |
2 |                                |
3 |                                |
4 |                                |
```

**Expected** (Driver cell): `4 passed in 0.01s` - the final state after all
four fixes.


- bug 1: resp read twice in, comment it (assume i deleted it)
- bug 2: raw query (city) passed through the url request,  must encode the query using urlencode
- bug 3: exception written in false sequence, it should be catching httperror then urlerror then timeout.
- bug 4: temperature key, to temp_c

In [5]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_weather.py", "-q", "-p", "no:cacheprovider"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr).replace("\r\n", "\n")
_out = re.sub(r" in \d+\.\d+s", " in 0.01s", _out)
print("\n".join(l for l in _out.splitlines() if "[100%]" not in l and l.strip()))

4 passed in 0.01s


## H2 - Build your own tiny API: the store

**Type:** REAL-WORLD - hard - writes 3 files

**Task**
Flip the client/server roles: build a small JSON store API **server** with
`http.server`, a **client** that talks to it, and tests that prove both. You
already know every piece - `send_response`/`send_header`/`end_headers`/`wfile`
from `local_api.py`, `Request` + `method="POST"` from M2, and categorized
errors from M1.

**File tree**

```
store_server.py       (you create)
store_client.py       (you create)
test_store.py         (you create)
```

**Interface contract**
Server routes:
- `GET  /items`         -> `{"count": n, "items": [...]}` (200)
- `GET  /items/{id}`    -> the item (200), or 404
- `POST /items` (JSON)  -> 201 with the stored item; 400 on bad input

`store_server.start(port=0)` -> a `ThreadingHTTPServer` on `127.0.0.1` (start
it inside a fixture; run `serve_forever` in a daemon thread).

Item shape: `{"id": int, "name": str, "price": float}`; ids handed out in
order starting at 1. Prices: reject anything not a number `> 0`. Names: reject
anything not a non-empty string.

Client contract:
- `list_items(base) -> list[dict]`
- `get_item(base, item_id) -> dict` - raises your own error carrying
  `status=404`
- `create_item(base, name, price) -> dict` - raises your own error carrying
  `status=400` on invalid input

**Constraints**
- TEST the server through the client as a normal user would - never through
  `requests`.
- Three tests: happy path (create one item, list it), POST validation (bad
  name and bad price -> 400), missing item (-> 404).
- Keep the server fixture in `test_store.py` (no `conftest.py` needed).
- Command: `python -m pytest test_store.py -q`

**Expected** (Driver cell): `3 passed in 0.01s`


In [11]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_store.py", "-q", "-p", "no:cacheprovider"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr).replace("\r\n", "\n")
_out = re.sub(r" in \d+\.\d+s", " in 0.01s", _out)
print("\n".join(l for l in _out.splitlines() if "[100%]" not in l and l.strip()))

3 passed in 0.01s


## Challenge - A CLI that caches API data

**Type:** CHALLENGE - hard - writes 2 files

**Task**
Real programs rarely hit the API for every screen. Build `cache_cli.py`, a
one-command CLI that fetches `/items`, merges them with an existing local cache
(offline-only items survive), and writes both `cache.json` and `cache.csv`.

**File tree**

```
cache_cli.py          (you create)
test_cache.py         (you create)
```

**Interface contract**
`python cache_cli.py --refresh`:
1. starts its own `local_api.LocalAPI()`,
2. fetches `/items` (the network set),
3. reads `cache.json` from the current directory if it exists (the cached set),
4. merges: every network id wins; cached-only items are kept,
5. writes `cache.json` (`{"items": [...]}`) and `cache.csv`
   (header `id,name,price,stock`),
6. prints exactly `Cache: N items (fresh X, cached Y)`.

**Constraints**
- Extract `merge(network, cached) -> (items, fresh, kept)` so the merge logic
  is testable without touching the network.
- Test 1: seed the cache with offline-only `{"id": 5, "name": "donut",
  "price": 1.0, "stock": 3}` -> merged ids are `[1,2,3,4,5]`, summary
  `Cache: 5 items (fresh 4, cached 1)`.
- Test 2: no existing cache -> summary `Cache: 4 items (fresh 4, cached 0)`.
- Run the CLI with `subprocess.run` using `cwd=tmp_path` and
  `env={**os.environ, "PYTHONPATH": <lesson folder>}` so it can import
  `local_api` while writing its files in the temp dir.
- Command: `python -m pytest test_cache.py -q`

**Expected** (Driver cell):
```
Cache: 5 items (fresh 4, cached 1)
2 passed in 0.01s
```


In [8]:
import json
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as td:
    tmp = Path(td)
    (tmp / "cache.json").write_text(
        json.dumps({"items": [{"id": 5, "name": "donut", "price": 1.0, "stock": 3}]}),
        encoding="utf-8",
    )
    env = {**os.environ, "PYTHONPATH": str(Path.cwd())}
    cli_path = str(Path.cwd() / "cache_cli.py")
    _r = subprocess.run(
        [sys.executable, cli_path, "--refresh"],
        cwd=tmp, env=env, capture_output=True, text=True,
    )
    _out = (_r.stdout + _r.stderr).replace("\r\n", "\n")
    print("\n".join(l for l in _out.splitlines() if l.strip()))

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_cache.py", "-q", "-p", "no:cacheprovider"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr).replace("\r\n", "\n")
_out = re.sub(r" in \d+\.\d+s", " in 0.01s", _out)
print("\n".join(l for l in _out.splitlines() if "[100%]" not in l and l.strip()))

Cache: 5 items (fresh 4, cached 1)
2 passed in 0.01s
